# 301 · Untrusted input experiment

This notebook goes with the article
[Untrusted input](https://leo-gan.github.io/GLD.SerializerBenchmark/theory/301/untrusted-input/).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/leo-gan/GLD.SerializerBenchmark/blob/master/docs/theory/notebooks/301/untrusted_input.ipynb)

Any public or multi-tenant endpoint that deserializes bytes should assume those bytes can be hostile.
Many incidents are not “JSON was a bit slow.” They are resource exhaustion or unsafe native deserialization.

You will wrap JSON parsing with hard limits on body size, nesting depth, and collection size,
and you will refuse Python pickle on the default boundary path.
This notebook does **not** demonstrate real exploits.

> **A note on numbers:** sizes and timings in these notebooks are only illustrations. For measured library comparisons on this project’s harness, use the suite [Results](https://leo-gan.github.io/GLD.SerializerBenchmark/) pages.


## Resource limits belong at the parser boundary

If you fully build a huge nested structure and only then validate it, you have already paid most of the denial-of-service cost.
The helpers here cap maximum bytes, maximum nesting depth, and maximum map or list size.

In a real service those limits should live at the trust boundary (the edge of the process that accepts untrusted input),
not only deep inside later business logic.


In [ ]:
import json
from typing import Any


class Limits:
    max_bytes = 10_000
    max_depth = 8
    max_collection = 100


def check_bytes(raw: bytes, lim: Limits = Limits()) -> None:
    if len(raw) > lim.max_bytes:
        raise ValueError(f"body too large: {len(raw)} > {lim.max_bytes}")


def check_depth(obj: Any, lim: Limits = Limits(), depth: int = 0) -> None:
    if depth > lim.max_depth:
        raise ValueError(f"nesting too deep: {depth}")
    if isinstance(obj, dict):
        if len(obj) > lim.max_collection:
            raise ValueError("too many map keys")
        for v in obj.values():
            check_depth(v, lim, depth + 1)
    elif isinstance(obj, list):
        if len(obj) > lim.max_collection:
            raise ValueError("too many array elements")
        for v in obj:
            check_depth(v, lim, depth + 1)


def parse_untrusted_json(raw: bytes, lim: Limits = Limits()) -> Any:
    check_bytes(raw, lim)
    obj = json.loads(raw)
    check_depth(obj, lim)
    return obj



## Ordinary JSON versus hostile shapes

The first payload is small and well formed; parsing should succeed.
Then you try three shapes that are cheap for an attacker and expensive for a worker:
very deep nesting, a map with far too many keys, and an oversized body.

Each hostile case should raise a clear error instead of locking up the runtime.


In [ ]:
ok = json.dumps({"user": "a", "items": [1, 2, 3]}).encode()
print("OK parse:", parse_untrusted_json(ok))

# depth bomb
depth = {"x": 0}
cur = depth
for _ in range(20):
    cur["n"] = {}
    cur = cur["n"]
deep = json.dumps(depth).encode()
try:
    parse_untrusted_json(deep)
    raise AssertionError("expected depth failure")
except ValueError as e:
    print("OK depth rejected:", e)

# cardinality bomb
wide = json.dumps({"k" + str(i): i for i in range(500)}).encode()
try:
    parse_untrusted_json(wide)
    raise AssertionError("expected cardinality failure")
except ValueError as e:
    print("OK cardinality rejected:", e)

# size bomb
big = b"{" + b'"a":"' + b"x" * 20_000 + b'"}'
try:
    parse_untrusted_json(big)
    raise AssertionError("expected size failure")
except ValueError as e:
    print("OK size rejected:", e)



## Never use native deserialization for untrusted bytes

Language-native formats such as pickle can reconstruct rich object graphs—and sometimes run code while doing so.
The boundary helper defaults to portable JSON with limits.
Pickle is allowed only when you explicitly opt in for a fully trusted demonstration.

A common real-world failure is a “debug” or “support” endpoint that still accepts pickle long after the public API was hardened.


In [ ]:
import pickle

def load_boundary(raw: bytes, allow_pickle: bool = False):
    if allow_pickle:
        # only for fully trusted same-process demos
        return pickle.loads(raw)
    # production default at untrusted boundary
    return parse_untrusted_json(raw)


trusted_local = pickle.dumps({"cache": True})
try:
    load_boundary(trusted_local, allow_pickle=False)
except Exception as e:
    print("OK pickle blocked at boundary:", type(e).__name__, e)

print("OK portable path:", load_boundary(ok, allow_pickle=False))



## Takeaways

Treat untrusted input as hostile by default.
Put size, depth, and cardinality limits at the boundary.
Fast results on friendly benchmark fixtures do not prove that a parser is safe under attack.

Related notebook: [Trust boundaries](./trust_boundaries.ipynb).
